In [ ]:
# 03 — Consolidation / Consolidación

**Project / Proyecto:** `excel-sales-consolidator`  
**Author / Autora:** Cristina  

---

**EN:** This notebook concatenates the 10 normalized DataFrames into a single consolidated DataFrame and exports it as a clean Excel file. Duplicate detection is included as a diagnostic step only — deep cleaning is delegated to the `excel-data-cleaner` module in the full pipeline.

**ES:** Este notebook concatena los 10 DataFrames normalizados en un único DataFrame consolidado y lo exporta como un archivo Excel limpio. La detección de duplicados se incluye solo como paso de diagnóstico — la limpieza profunda se delega al módulo `excel-data-cleaner` en el pipeline completo.

---

**Input:** `data/raw/source_01.xlsx` → `data/raw/source_10.xlsx`  
**Output:** `outputs/consolidated.xlsx`

## 0 — Setup / Configuración

**EN:** Import libraries, reload source files, and apply normalization to produce `lista_dfs_normalized`.  
**ES:** Importar librerías, recargar archivos fuente y aplicar normalización para producir `lista_dfs_normalized`.

In [2]:
import pandas as pd
import sys
from pathlib import Path
from datetime import datetime

print("Libraries loaded successfully")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# Resolve root path — works whether notebook runs from /notebooks or project root
ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUTPUT_FILE = ROOT / "outputs" / "consolidated.xlsx"

print(f"Project root: {ROOT}")
print(f"Output path:  {OUTPUT_FILE}")

Libraries loaded successfully
Timestamp: 2026-04-04 19:39:10

Project root: C:\Users\crisr\dev\PORTFOLIO\excel-sales-consolidator
Output path:  C:\Users\crisr\dev\PORTFOLIO\excel-sales-consolidator\outputs\consolidated.xlsx


In [3]:
# Load all 10 source files into a list of raw DataFrames

lista_excel_files = [f"source_{i:02d}.xlsx" for i in range(1, 11)]

lista_dfs =[]

for f in lista_excel_files:
    INPUT_FILE = ROOT / "data" / "raw" / f
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}") 
        
    df = pd.read_excel(INPUT_FILE, dtype=str)  # Read all columns as string to avoid early type coercion
    df.columns = df.columns.str.strip()        # Strip whitespace from column names immediately on load
    lista_dfs.append(df)

print(f"{len(lista_dfs)} files loaded successfully")   

10 files loaded successfully


In [4]:
# Column map — 53 unique variants mapped to 12 standard column names
COLUMN_MAP = {
    "id_venta":      ["Venta ID", "ID_VENTA", "ID Venta", "Id venta"],
    "fecha_venta":   ["fecha venta", "fecha_venta_alt", "Fecha_Venta", "fecha operación",
                      "fecha", "fecha_venta", "FECHA_VENTA", "Fecha Venta"],
    "producto":      ["Producto", "tipo_producto", "articulo", "producto"],
    "tipo_madera":   ["TIPO_MADERA", "tipo_madera", "madera", "tipo madera"],
    "certificacion": ["certif", "certificacion", "certificación"],
    "cliente":       ["CLIENTE", "cliente", "Cliente", "cliente_2", "cliente_alt",
                      "NOMBRE CLIENTE", "razon_social"],
    "cantidad_m3":   ["Cantidad m3", "cantidad_m3", "m3", "volumen_m3"],
    "precio_m3":     ["€/m3", "precio m3", "precio_m3", "Precio_m3"],
    "importe":       ["importe", "importe_total", "total"],
    "estado":        ["Estado", "estado", "situacion", "status"],
    "comercial":     ["comercial", "responsable_comercial", "sales_rep", "vendedor"],
    "pais":          ["country", "mercado", "pais", "País"],
}


def normalize_df(df: pd.DataFrame, column_map: dict, source_name: str) -> pd.DataFrame:
    """
    EN: Maps raw column variants to standard names and keeps only relevant columns.
    ES: Mapea variantes de columnas al nombre estándar y conserva solo las relevantes.
    """
    out = pd.DataFrame()

    for standard_col, variants in column_map.items():
        found = [v for v in variants if v in df.columns]

        if not found:
            out[standard_col] = pd.NA
        elif len(found) == 1:
            out[standard_col] = df[found[0]]
        else:
            # Multiple variants — combine prioritizing first non-null value per row
            combined = df[found[0]]
            for col in found[1:]:
                combined = combined.combine_first(df[col])
            out[standard_col] = combined

    # Add traceability column to track which source file each row came from
    out["source_file"] = source_name

    return out.reset_index(drop=True)


# Apply normalization to all 10 DataFrames
lista_dfs_normalized = []

for i, df in enumerate(lista_dfs, start=1):
    source_name = f"source_{i:02d}.xlsx"
    df_norm = normalize_df(df, COLUMN_MAP, source_name)
    lista_dfs_normalized.append(df_norm)

print(f"{len(lista_dfs_normalized)} DataFrames normalized successfully")

10 DataFrames normalized successfully


## 1 — Consolidation / Consolidación

**EN:** Concatenate all 10 normalized DataFrames into a single DataFrame using `pd.concat()`. The index is reset to produce a clean sequential row index.  
**ES:** Concatenar los 10 DataFrames normalizados en un único DataFrame usando `pd.concat()`. El índice se resetea para producir un índice de filas secuencial limpio.

In [5]:
df_consolidated = pd.concat(lista_dfs_normalized, ignore_index=True)

print(f"Consolidated DataFrame: {df_consolidated.shape[0]} rows × {df_consolidated.shape[1]} columns")
display(df_consolidated.head(10))

Consolidated DataFrame: 401 rows × 13 columns


,id_venta,fecha_venta,producto,tipo_madera,certificacion,cliente,cantidad_m3,precio_m3,importe,estado,comercial,pais,source_file
0,1102,02/01/2024,CLT,abeto,FSC,Muebles Pérez Polo,NaN,EUR 320.00,NaN,cerrado,Juan,Madrid,source_01.xlsx
1,1270,NaN,Viga laminada,Abeto,NaN,REFORMAS LÓPEZ,"0,00",EUR 450.00,NaN,ok,Juan,ES,source_01.xlsx
2,1106,NaN,Viga,PINO,PEFC,Muebles Pérez Polo,"2,00",EUR 450.00,"900,00 €",pendiente,Juan,ES,source_01.xlsx
3,1071,2024-02-06,Viga,PINO,FSC,Promociones Sur,"1,50",EUR 450.00,675,CERRADO,Juan,ES,source_01.xlsx
4,1188,02/01/2024,Viga,abeto,CE,Muebles Pérez Polo,"1,50",EUR 320.00,NaN,ok,Ana,España,source_01.xlsx
5,1020,NaN,Viga,PINO,FSC,Reformas López,"0,00",EUR 320.00,NaN,pendiente,Juan,ES,source_01.xlsx
6,1102,02/01/2024,CLT,abeto,FSC,Muebles Pérez Polo,NaN,EUR 320.00,NaN,cerrado,Juan,Madrid,source_01.xlsx
7,1121,NaN,Viga laminada,pino,FSC,Construcciones Norte,"2,00",EUR 320.00,640,ok,Juan,España,source_01.xlsx
8,1214,2024-02-06,Viga,Pino,PEFC,Muebles Pérez Polo,"1,50",EUR 450.00,675,cerrado,Ana,Madrid,source_01.xlsx
9,1087,02/01/2024,CLT,abeto,NaN,NaN,"2,00",EUR 320.00,NaN,cerrado,Ana,ES,source_01.xlsx


## 2 — Diagnosis / Diagnóstico

**EN:** Inspect the consolidated DataFrame — missing values, duplicate rows, and row count per source file. Duplicates are flagged here but not removed: deep cleaning is handled by `excel-data-cleaner` in the full pipeline.  
**ES:** Inspeccionar el DataFrame consolidado — valores nulos, filas duplicadas y número de filas por archivo fuente. Los duplicados se detectan aquí pero no se eliminan: la limpieza profunda la realiza `excel-data-cleaner` en el pipeline completo.

In [6]:
# Missing values per column
print("Missing values per column:")
print(df_consolidated.isna().sum())
print(f"\nTotal missing values: {df_consolidated.isna().sum().sum()}")

Missing values per column:
id_venta          20
fecha_venta      218
producto          20
tipo_madera       20
certificacion    104
cliente          109
cantidad_m3       72
precio_m3         20
importe          176
estado            20
comercial         20
pais              20
source_file        0
dtype: int64

Total missing values: 819


In [7]:
# Duplicate rows — diagnostic only, not removed
n_duplicates = df_consolidated.duplicated().sum()
print(f"Duplicate rows detected: {n_duplicates}")
print("Note: duplicates are not removed here — delegated to excel-data-cleaner in the pipeline")

Duplicate rows detected: 25
Note: duplicates are not removed here — delegated to excel-data-cleaner in the pipeline


In [8]:
# Row count per source file — confirms traceability column works correctly
print("Rows per source file:")
print(df_consolidated["source_file"].value_counts().sort_index())
print(f"\nTotal rows: {len(df_consolidated)}")

Rows per source file:
source_file
source_01.xlsx    31
source_02.xlsx    48
source_03.xlsx    34
source_04.xlsx    55
source_05.xlsx    41
source_06.xlsx    27
source_07.xlsx    44
source_08.xlsx    39
source_09.xlsx    32
source_10.xlsx    50
Name: count, dtype: int64

Total rows: 401


## 3 — Export / Exportación

**EN:** Export the consolidated DataFrame to `outputs/consolidated.xlsx`. No formatting applied — visual styling is handled by `excel-report-formatter` in the full pipeline.  
**ES:** Exportar el DataFrame consolidado a `outputs/consolidated.xlsx`. Sin formato visual — el estilo lo aplica `excel-report-formatter` en el pipeline completo.

In [9]:
# Create outputs directory if it doesn't exist
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_consolidated.to_excel(OUTPUT_FILE, index=False, engine="openpyxl")

print(f"✓ File exported: {OUTPUT_FILE.name}")
print(f"  Shape : {df_consolidated.shape[0]} rows × {df_consolidated.shape[1]} columns")
print(f"  Path  : {OUTPUT_FILE}")

✓ File exported: consolidated.xlsx
  Shape : 401 rows × 13 columns
  Path  : C:\Users\crisr\dev\PORTFOLIO\excel-sales-consolidator\outputs\consolidated.xlsx


---

## Summary / Resumen

**EN:**
- 10 normalized DataFrames concatenated into a single consolidated DataFrame
- `source_file` column confirms traceability per row
- Duplicate rows detected and flagged — not removed (delegated to `excel-data-cleaner`)
- Output exported to `outputs/consolidated.xlsx` without visual formatting
- → Next step: `excel-data-cleaner` for deep cleaning, then `excel-report-formatter` for styling

**ES:**
- 10 DataFrames normalizados concatenados en un único DataFrame consolidado
- La columna `source_file` confirma la trazabilidad por fila
- Filas duplicadas detectadas y registradas — no eliminadas (delegado a `excel-data-cleaner`)
- Resultado exportado a `outputs/consolidated.xlsx` sin formato visual
- → Siguiente paso: `excel-data-cleaner` para limpieza profunda, luego `excel-report-formatter` para el estilo